# Onechess : menaces étroites sans défenses contre témoin (TPU Colab v5e-1)

Exécution : Environnement d'exécution > Modifier le type > **TPU v5e-1**, puis Exécution > Tout exécuter. Garder l'onglet ouvert (~2 h 30). Le résultat est la ligne `BILAN` de chaque variante (erreur de validation en cp, la plus basse gagne).

In [ ]:
!python -c "import jax; print(jax.devices())"

In [ ]:
import io, tarfile, urllib.request, glob
tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen('https://codeload.github.com/ovnice-dev/onechess-tools/tar.gz/main').read())).extractall('/content/code')
TOOLS = glob.glob('/content/code/*/tools')[0]
INIT = glob.glob('/content/code/*/reseau/archives/final_v1.0.pt')[0]
print(TOOLS, INIT)
!pip -q install optax chess
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!python -u {TOOLS}/hf_sources.py --url https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/nodes5000pv2_UHO.binpack --prefixe n5 --dest /content/donnees --octets 2000000000 --garde 1.0 --fixe 240 --graine 11 --parts 2
import numpy as np, os
v = sorted(glob.glob('/content/donnees/*_p00_*.bin'))[0]
os.rename(v, '/content/donnees/z_val.bin')
np.fromfile('/content/donnees/z_val.bin', dtype='uint8').reshape(-1, 32)[:2_000_000].tofile('/content/donnees/z_val.bin')
print(len(glob.glob('/content/donnees/*.bin')), 'fichiers')

In [ ]:
# Temoin : deja fait (64,4 -> 63,0 cp). Ne relancer que si la session a ete perdue.
COMMUN = f"{TOOLS}/train_jax.py --stream '/content/donnees/*.bin' --hidden 1024 --buckets mat4 --out-buckets 4 --chunk-files 4 --epoch-files 16 --init {INIT} --lr 3e-4 --val 2000000 --ft-dense 1 --epochs 10 --limite-s 5400"
# !python -u {COMMUN} --out /content/temoin.nnue 2>&1 | grep -v '^ *$' | tail -15

In [ ]:
!python -u {COMMUN} --menaces 1 --menaces-largeur 256 --menaces-ennemis 1 --out /content/etroit_ennemis.nnue 2>&1 | grep -v '^ *$' | tail -15